In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ---- input / output ----
INTERGENIC_CSV = "/rhome/zli529/lab/LncRNA_chip_prediction/NEW/intergenic_regions.csv"
OUT_CSV        = "intergenic_midpoints_100bp.csv"

STEP = 100  # bp

# ---- load intergenic ----
df = pd.read_csv(INTERGENIC_CSV)
df.columns = [c.strip().lower() for c in df.columns]

# normalize columns to chr, start, end
if "chr" not in df.columns:
    df.rename(columns={df.columns[0]: "chr"}, inplace=True)

if "start" not in df.columns or "end" not in df.columns:
    raise ValueError("intergenic_regions.csv must have columns: chr, start, end")

df["start"] = df["start"].astype(int)
df["end"]   = df["end"].astype(int)
df["start"], df["end"] = np.minimum(df["start"], df["end"]), np.maximum(df["start"], df["end"])

# ---- generate mids ----
rows = []
mid_id = 1

for row in tqdm(df.itertuples(index=False), total=len(df), desc="Intergenic regions"):
    chrom = getattr(row, "chr")
    start = int(getattr(row, "start"))
    end   = int(getattr(row, "end"))

    # forward: start -> end, step 100, include end
    mids_fwd = list(range(start, end + 1, STEP))
    if mids_fwd[-1] != end:
        mids_fwd.append(end)

    for mid in mids_fwd:
        rows.append((chrom, mid, mid,mid_id, "+"))
        mid_id += 1

    # backward: end -> start, step 100, include start
    mids_rev = list(range(end, start - 1, -STEP))
    if mids_rev[-1] != start:
        mids_rev.append(start)

    for mid in mids_rev:
        rows.append((chrom, mid, mid, mid_id, "-"))
        mid_id += 1

# ---- save ----
out = pd.DataFrame(rows, columns=["chr", "mid", "mid", "mid_ID", "strand"])
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}\nRows: {len(out):,}")
out.head()
